In [14]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")
    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")
    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]
def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing lab6_search.py


In [16]:
%%writefile guided_chatbot.py
from lab6_search import Retriever
FAQ = [
    {"id": "faq:rank", "question": "Can Superior rank?",
     "answer": "Judging by Superior behavior, never.",
     "source": "campus_notes.txt"},
    {"id": "faq:road",
     "question": "Is there a road in Superior or is Superior itself on the road?",
     "answer": "Brother, Superior itself is on the road.",
     "source": "campus_notes.txt"},
    {"id": "faq:admission",
     "question": "Is Superior famous enough to take admission there?",
     "answer": "Oh brother, it is not famous at all. Take admission somewhere else.",
     "source": "campus_notes.txt"},
    {"id": "faq:timetable_app",
     "question": "Why didn't Superior buy Sir Rasikh timetable app?",
     "answer": "Superior did not have that much budget & Sir demanded a lot.",
     "source": "campus_notes.txt"},
    {"id": "faq:world_map",
     "question": "Will Sir Rasikh build the full 3D world map solution?",
     "answer": "Why not, Sir Rasikh will do it. He is just not in form right now.",
     "source": "campus_notes.txt"}]

def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)
def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    # Only exact canonical FAQ text is answered automatically here.
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"],
                        default="tfidf")
    args = parser.parse_args()
    engine = build_bot(args.backend)
    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            query = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        print("Bot:", reply(engine, query))

Overwriting guided_chatbot.py


In [17]:
!ls

guided_chatbot.py     guided_suggestions.py  sample_data
guided_chunks.py      lab6_search.py	     task2_chunks.py
guided_embeddings.py  __pycache__	     task2_compare.py


In [2]:
%%writefile task2_chunks.py
import re
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result
def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    # Teaching splitter: punctuation followed by whitespace.
    cuts = [0] + [m.end() for m in
                  re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out
def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = (sentence_chunks(text) if strategy == "sentence"
                 else windows(text, 40, int(strategy)))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id, start=a, end=b, text=text[a:b]))
    return result
if __name__ == "__main__":
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")
    long = " ".join(["word"] * 95) + "."
    assert all(len(long[a:b].split()) <= 40
               for a, b in sentence_chunks(long))
    docs = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ["0", "10", "sentence"]:
        for c in make_chunks(docs, strategy):
            assert c["text"] == docs[c["doc"]][c["start"]:c["end"]]
            print(strategy, c)

Writing task2_chunks.py


In [3]:
!python task2_chunks.py

0 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
10 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
sentence {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}


In [4]:
%%writefile guided_chunks.py
from task2_chunks import windows, sentence_chunks
text = ("Late work is penalized. Illness may qualify for an extension "
        "when medical evidence is provided. Ask the coordinator first.")
for name, spans in [
        ("word windows", windows(text, size=12, overlap=4)),
        ("sentence aware", sentence_chunks(text, size=12))]:
    print(name)
    for start, end in spans:
        print(start, end, repr(text[start:end]))
        assert 0 <= start < end <= len(text)

Writing guided_chunks.py


In [5]:
!python guided_chunks.py

word windows
0 73 'Late work is penalized. Illness may qualify for an extension when medical'
48 122 'an extension when medical evidence is provided. Ask the coordinator first.'
sentence aware
0 24 'Late work is penalized. '
24 96 'Illness may qualify for an extension when medical evidence is provided. '
96 122 'Ask the coordinator first.'


In [6]:
%%writefile task2_compare.py
from task2_chunks import make_chunks, windows, sentence_chunks

DOCS = {
    "late": (
        "Assessment Policy. Assignments are due at 11:59 PM on the date "
        "stated in the course outline. Late submission is normally "
        "penalized by ten percent for each day after the deadline. "
        "Illness extensions require medical evidence before the deadline. "
        "Students must send the certificate to the course coordinator and "
        "wait for a written reply before assuming that an extension has "
        "been granted. Extensions are not given for travel plans or for a "
        "heavy workload in other courses. Group projects follow the same "
        "rule, and each member must report any illness separately to the "
        "coordinator. When a student submits work after the final accepted "
        "date, the coordinator records the submission time, compares it "
        "with the approved extension if there is one, applies the penalty "
        "for every full day of delay, and sends the final mark to the "
        "student together with a short note explaining how the penalty "
        "was calculated. Students may ask for a review of the penalty "
        "within five working days. Marks are published on the learning "
        "portal after all submissions have been checked."),
    "password": (
        "IT Help Guide. Every student receives one university account "
        "that is used for email, the learning portal and the library. "
        "Reset a forgotten password using the recovery link. The link is "
        "shown on the login page and is sent to the registered phone "
        "number or the alternate email address. Never share recovery "
        "codes with anyone, including friends, classmates or staff. "
        "Accounts locked after five failed attempts cannot use the "
        "recovery link and must be unlocked by the help desk. The help "
        "desk asks for the student card and a short description of the "
        "problem, and it does not ask for the current password at any "
        "time. After a successful reset the student should log out of "
        "all shared computers, choose a new password that has at least "
        "twelve characters, and avoid using the same password for "
        "personal accounts outside the university. Passwords expire "
        "every one hundred and eighty days, and a reminder is sent "
        "fourteen days before the expiry date."),
    "assignment": (
        "Submission Guide. Each course uses the same method for "
        "handing in written work so that files are never lost. Upload "
        "assignments through the learning portal. Submit the final file "
        "as a PDF and write the course code and student number in the "
        "file name. Students who cannot access the portal may email the "
        "file to the coordinator before the deadline and must keep the "
        "sent message as proof. The portal accepts files up to twenty "
        "megabytes, and larger files should be compressed or split into "
        "two parts that are clearly numbered. Each upload creates a "
        "receipt that shows the date and time, and students should save "
        "this receipt until the final marks are published. Only the "
        "latest upload before the deadline is marked, so a student may "
        "replace an earlier file as many times as needed. Source code, "
        "data files and images must be placed in one folder and "
        "submitted as a single archive together with the report. "
        "Plagiarism checks run automatically on every upload."),
}
PAIRS = {
    "late": ("Late submission is normally penalized by ten percent for "
             "each day after the deadline.",
             "Illness extensions require medical evidence before the "
             "deadline."),
    "password": ("Reset a forgotten password using the recovery link.",
                 "Accounts locked after five failed attempts cannot use "
                 "the recovery link and must be unlocked by the help desk."),
    "assignment": ("Upload assignments through the learning portal.",
                   "Students who cannot access the portal may email the "
                   "file to the coordinator before the deadline"),
}
if __name__ == "__main__":
    original_words = {d: len(t.split()) for d, t in DOCS.items()}
    for d, t in DOCS.items():
        longest = max(len(s.split()) for s in t.replace("?", ".").split("."))
        print(d, "words:", original_words[d],
              "| longest sentence (words):", longest)
        assert d in PAIRS and all(p in t for p in PAIRS[d])
    total_original = sum(original_words.values())
    print()
    for strategy, name in [("0", "window 40, overlap 0"),
                           ("10", "window 40, overlap 10"),
                           ("sentence", "sentence-aware 40")]:
        chunks = make_chunks(DOCS, strategy)
        sizes = [len(c["text"].split()) for c in chunks]
        duplicate = sum(sizes) - total_original
        survived = 0
        for d, (rule, exc) in PAIRS.items():
            if any(c["doc"] == d and rule in c["text"] and exc in c["text"]
                   for c in chunks):
                survived += 1
        print(f"{name}: chunks={len(chunks)}, "
              f"avg words={sum(sizes) / len(sizes):.1f}, "
              f"min={min(sizes)}, max={max(sizes)}, "
              f"duplicate words={duplicate}, "
              f"rule+exception in one chunk={survived}/{len(PAIRS)}")
        for c in chunks:
            assert c["text"] == DOCS[c["doc"]][c["start"]:c["end"]]
            print("   ", c["id"], c["start"], c["end"],
                  len(c["text"].split()), "words")
        print()
    print("Boundary tests")
    print("empty text:", windows(""), sentence_chunks(""))
    print("short document:", windows("short text"),
          sentence_chunks("short text"))
    try:
        windows("a b", 2, 2)
    except ValueError as error:
        print("invalid overlap -> ValueError:", error)
    long = " ".join(["word"] * 95) + "."
    spans = sentence_chunks(long)
    print("95-word sentence ->", len(spans), "chunks of",
          [len(long[a:b].split()) for a, b in spans], "words")

Writing task2_compare.py


In [7]:
!python task2_compare.py

late words: 173 | longest sentence (words): 54
password words: 160 | longest sentence (words): 35
assignment words: 165 | longest sentence (words): 24

window 40, overlap 0: chunks=14, avg words=35.6, min=5, max=40, duplicate words=0, rule+exception in one chunk=1/3
    late:0 0 257 40 words
    late:1 258 495 40 words
    late:2 496 746 40 words
    late:3 747 968 40 words
    late:4 969 1052 13 words
    password:0 0 224 40 words
    password:1 225 479 40 words
    password:2 480 697 40 words
    password:3 698 942 40 words
    assignment:0 0 232 40 words
    assignment:1 233 460 40 words
    assignment:2 461 694 40 words
    assignment:3 695 922 40 words
    assignment:4 923 957 5 words

window 40, overlap 10: chunks=17, avg words=37.5, min=15, max=40, duplicate words=140, rule+exception in one chunk=1/3
    late:0 0 257 40 words
    late:1 179 434 40 words
    late:2 377 615 40 words
    late:3 558 797 40 words
    late:4 747 968 40 words
    late:5 915 1052 23 words
    password:0

In [8]:
%%writefile guided_embeddings.py
import numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
passages = ["Reset a forgotten password using the recovery link.",
            "Submit assignments through the learning portal.",
            "Check ink levels when a printer produces blank pages."]
C = model.encode(passages, normalize_embeddings=True)
Q = model.encode(["How can I recover my login?"],
                 normalize_embeddings=True)
scores = (Q @ C.T)[0]
print("Embedding shape:", C.shape)
for i in np.argsort(-scores):
    print(i, round(float(scores[i]), 3), passages[i])
assert np.allclose(np.linalg.norm(C, axis=1), 1, atol=1e-5)

Writing guided_embeddings.py


In [9]:
!python guided_embeddings.py

modules.json: 100% 349/349 [00:00<00:00, 1.27MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 452kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 21.2MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 220kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.36MB/s]

model.safetensors: downloading bytes:  79% 71.6M/90.9M [00:00<00:00, 113MB/s, 5.07MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 72.7MB/s, 8.18MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 77.7MB/s, 8.85MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 4941.76it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.33MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 14.4MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 40.2MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 429kB/s]
config.json: 100% 190/190 [00:00<00:00, 698kB/s]
Embedding shape: (3, 384)
0 0.627 Reset a forgotten password using the recovery link.
1 0.072 Submit as

In [10]:
%%writefile guided_suggestions.py
from guided_chatbot import build_bot
engine = build_bot("embedding")
query = "superior admission"
print("Suggested questions:")
for record, score in engine.search(query, k=3):
    print(record["question"], f"({score:.3f})")

Writing guided_suggestions.py


In [18]:
!python guided_suggestions.py

Loading weights: 100% 103/103 [00:00<00:00, 4246.25it/s]
Suggested questions:
Is Superior famous enough to take admission there? (0.664)
Is there a road in Superior or is Superior itself on the road? (0.484)
Can Superior rank? (0.460)


In [19]:
%%writefile task2_semantic_check.py
from task2_chunks import make_chunks
from task2_compare import DOCS
from lab6_search import Retriever
records = make_chunks(DOCS, "sentence")
queries = [
    "How can I regain access to my account?",
    "How should I hand in my coursework?",
    "What proof is needed when I was ill?",
]
for backend in ["tfidf", "embedding"]:
    engine = Retriever(records, backend)
    print("=" * 60)
    print("Backend:", backend)
    for query in queries:
        print("\nQuery:", query)
        hits = engine.search(query, k=2)
        if not hits:
            print("  No matching records.")
        for r, score in hits:
            print(f"  {r['id']} score={score:.3f} "
                  f"chars {r['start']}-{r['end']}")
            print("   ", r["text"])

Writing task2_semantic_check.py


In [20]:
!python task2_semantic_check.py

Backend: tfidf

Query: How can I regain access to my account?
  late:4 score=0.254 chars 813-897
    to the student together with a short note explaining how the penalty was calculated.
  assignment:2 score=0.197 chars 251-388
    Students who cannot access the portal may email the file to the coordinator before the deadline and must keep the sent message as proof. 

Query: How should I hand in my coursework?
  late:4 score=0.235 chars 813-897
    to the student together with a short note explaining how the penalty was calculated.
  assignment:4 score=0.108 chars 526-661
    Each upload creates a receipt that shows the date and time, and students should save this receipt until the final marks are published. 

Query: What proof is needed when I was ill?
  assignment:5 score=0.168 chars 661-781
    Only the latest upload before the deadline is marked, so a student may replace an earlier file as many times as needed. 
  late:4 score=0.161 chars 813-897
    to the student together with a s